# 75. KAMP 사례 참고 정상 오토인코더·노이즈 제거 AE 앙상블
참고상품668은D사100/180T다른데이터이고가스분류성적을공개하지않는다. 이노트북은정상라벨만학습해x→x복원한다.0/.05노이즈AE각2seed,선형PCA기준도비교한다. PyTorch설치없이기존sklearn MLPRegressor로진짜병목복원망을구현한다.
제품별정상기준은외부참조학습에서만fit. AE학습·스케일·임계값·내부선택에서평가구간정답제외. 라벨없는자료를모두정상이라고가정하지않는다. 이미분석한개발자료의중첩검증이지새표본독립평가아니다. 복원오차이상은가스정답이아니다.


In [ ]:
from pathlib import Path
import json,warnings,numpy as np,pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from sklearn.exceptions import ConvergenceWarning
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/74_product_normalized_joint_gas_modes.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.rsplit('memo={}',1)[0])
memo={};logs=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 per_view=[]
 for view,(tr,te) in enumerate(views(ref,q)):
  tr=np.clip(tr,-8,8);te=np.clip(te,-8,8);normal=tr[y[ref]==0]
  ae_scores=[]
  for noise in [0.,.05]:
   predictions=[];normal_errors=[]
   for seed in [42,7]:
    rng=np.random.default_rng(seed)
    noisy=normal+rng.normal(0,noise,normal.shape)
    m=MLPRegressor(hidden_layer_sizes=(8,2 if view==0 else 3,8),activation='tanh',alpha=.01,learning_rate_init=.005,max_iter=200,early_stopping=True,validation_fraction=.15,n_iter_no_change=15,random_state=seed,batch_size=min(64,len(normal)))
    with warnings.catch_warnings():
     warnings.simplefilter('ignore',ConvergenceWarning)
     m.fit(noisy,normal)
    error=np.mean((m.predict(te)-te)**2,axis=1)
    cal=np.mean((m.predict(normal)-normal)**2,axis=1)
    unit=max(float(np.quantile(cal,.95)),1e-6)
    predictions.append(error/unit)
    logs.append(dict(train=len(ref),query=len(q),view=view,noise=noise,seed=seed,normal_training=len(normal),iterations=m.n_iter_,converged_before_limit=bool(m.n_iter_<200)))
   ae_scores.append(np.mean(predictions,axis=0))
  pca=PCA(n_components=2 if view==0 else 3,random_state=42).fit(normal)
  error=np.mean((pca.inverse_transform(pca.transform(te))-te)**2,axis=1)
  trainerror=np.mean((pca.inverse_transform(pca.transform(normal))-normal)**2,axis=1)
  ae_scores.append(error/max(float(np.quantile(trainerror,.95)),1e-6))
  per_view.append(np.column_stack(ae_scores))
 memo[key]=np.max(per_view,axis=0)
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 outer=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['normal_AE','denoising_normal_AE','linear_PCA_AE']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   ip=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],ip,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,outer[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(outer[j,k]>cut),score=float(outer[j,k]),threshold=cut))
 selected=max(candidates,key=lambda a:a[:3]);choices.append(dict(block=int(b),selected=selected[3],budget=selected[4],inner_recall=selected[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(selected[5][j]),score=np.nan,threshold=np.nan))
 print('Finished normal-autoencoder outer block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget,product),g in p.groupby(['model','budget','product']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(logs),'training_audit')]:
 obj.to_csv(tables/f'gas_normal_autoencoder_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('AE reconstruction targets are sensor values, NOT pseudo gas ground truth. Known-data nested evaluation, not fresh independent validation.')
